# Veri Seti ve Gerçek Etiketler — Nesli

- 6 bağımsız görüntü: **G1–G2 geliştirme**, **K1–K4 kontrol**. Dönüştürülmüş kopya bağımsız örnek sayılmaz.
- Doğru sınıf, yöntemler çalıştırılmadan **önce** elle belirlenir: başta baret / baret var, başta değil / baret yok / belirsiz.
- Baş bölgesi ve baret kutuları elle işaretlenir (fırça maskesi → `betikler/etiketleri_uret.py`). Görünmeyen baş için etiket uydurulmaz.

In [ ]:
from pathlib import Path
import sys
KOK = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kaynak" / "analiz.py").is_file())
sys.path.insert(0, str(KOK))

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from kaynak.veri import ayarlari_oku, veri_setini_oku, goruntu_oku, maske_oku
from kaynak.analiz import analiz_et
from kaynak.gorsel import bgr2rgb, kutulari_ciz, maske_bindir
ayarlar = ayarlari_oku()
kayitlar = veri_setini_oku()
pd.set_option("display.max_colwidth", 80)
print("Görüntüler:", ", ".join(f"{k['kimlik']} ({k['bolum']})" for k in kayitlar))

## 1. Kaynak listesi

In [ ]:
pd.DataFrame(kayitlar)[["kimlik", "bolum", "genislik", "yukseklik", "kosul", "kaynak_url", "lisans"]]

## 2. Önceden belirlenen sınıflar ve gerekçeler

In [ ]:
pd.DataFrame(kayitlar)[["kimlik", "bolum", "dogru_sinif", "gerekce", "bas_roi", "baret_kutulari"]]

## 3. Etiketlerin görüntü üzerinde kontrolü
Yeşil: baret kutusu · Gri: baş ROI · Sarı: fırçayla boyanmış baret maskesi (varsa)

In [ ]:
fig, eksen = plt.subplots(2, 3, figsize=(18, 10))
for e, k in zip(eksen.ravel(), kayitlar):
    g = goruntu_oku(k["dosya"])
    m = maske_oku(k["kimlik"], "baret")
    if m is not None:
        g = maske_bindir(g, m)
    g = kutulari_ciz(g, [], k["baret_kutulari"], {"eslesmeler": []}, k["bas_roi"])
    e.imshow(bgr2rgb(g)); e.set_title(f"{k['kimlik']} — {k['dogru_sinif']}"); e.axis("off")
plt.tight_layout(); plt.show()

## 4. Belirsizlik notu
Baş görünmeyen ya da karar için kanıtın yetersiz olduğu görüntülerde sınıf **belirsiz** seçilir; örtülü baş "baret yok" diye etiketlenmez.